# Notebook 5 - IA Générative : Rapport RH Automatique
## UCAO 2025-2026 | Seye Kiné & Bindia Adeline Thiara | M. Aidara

Objectif : Générer automatiquement un rapport RH narratif en français à partir des résultats du modèle ML.

---
## Cellule 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")
print("OK")

OK


---
## Cellule 2 — Chargement du modèle

In [2]:
import joblib
data = joblib.load("mon_modele_rh.pkl")

modele        = data["cerveau_ia"]
preprocesseur = data["traitement"]
seuil         = data["reglage_seuil"]
FEATURES      = data["features"]
f1_final      = data["f1"]
auc_final     = data["auc"]

df = pd.read_csv("hr.csv")
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})
X_all             = preprocesseur.transform(df[FEATURES])
df["Probabilite"] = modele.predict_proba(X_all)[:, 1]
df["Prediction"]  = (df["Probabilite"] >= seuil).astype(int)
df["Niveau"]      = df["Probabilite"].apply(lambda p:
    "Critique" if p >= 0.70 else "Eleve" if p >= 0.50 else "Modere" if p >= seuil else "Faible")

print(f"OK — {len(df):,} employes charges")

OK — 1,470 employes charges


---
## Cellule 3 — Statistiques

In [3]:
n_total  = len(df)
n_part   = df["Attrition"].sum()
taux     = n_part / n_total * 100
n_crit   = (df["Niveau"] == "Critique").sum()
n_eleve  = (df["Niveau"] == "Eleve").sum()
n_modere = (df["Niveau"] == "Modere").sum()
n_faible = (df["Niveau"] == "Faible").sum()

dept_risque = df.groupby("Department")["Probabilite"].mean() * 100
dept_max    = dept_risque.idxmax()
dept_min    = dept_risque.idxmin()

ot_yes = df[df["OverTime"]=="Yes"]["Attrition"].mean() * 100
ot_no  = df[df["OverTime"]=="No"]["Attrition"].mean() * 100
date_rapport = datetime.now().strftime("%d/%m/%Y a %H:%M")

print("Statistiques calculees :")
print(f"  Total : {n_total}  |  Partis : {n_part} ({taux:.1f}%)")
print(f"  Critique : {n_crit}  |  Eleve : {n_eleve}  |  Modere : {n_modere}  |  Faible : {n_faible}")
print(f"  Dept le plus risque : {dept_max} ({dept_risque[dept_max]:.1f}%)")

Statistiques calculees :
  Total : 1470  |  Partis : 237 (16.1%)
  Critique : 200  |  Eleve : 6  |  Modere : 50  |  Faible : 1214
  Dept le plus risque : Sales (20.1%)


---
## Cellule 4 — Génération du rapport narratif

In [4]:
# Construction du rapport ligne par ligne
lignes = []
lignes.append("=" * 70)
lignes.append("  RAPPORT RH — ANALYSE PREDICTIVE DU TURNOVER")
lignes.append(f"  Genere automatiquement le {date_rapport}")
lignes.append("=" * 70)
lignes.append("")
lignes.append("1. RESUME EXECUTIF")
lignes.append("-" * 70)
lignes.append(f"Analyse realisee sur {n_total:,} employes.")
lignes.append(f"Taux d attrition observe : {taux:.1f}% ({n_part} departs).")
lignes.append(f"Modele XGBoost : F1={f1_final:.4f} | AUC={auc_final:.4f}")
lignes.append("")
lignes.append("2. REPARTITION DES NIVEAUX DE RISQUE")
lignes.append("-" * 70)
lignes.append(f"  Risque Critique (>=70%) : {n_crit:>5,} employes — Action immediate")
lignes.append(f"  Risque Eleve   (50-70%) : {n_eleve:>5,} employes — Entretien sous 2 semaines")
lignes.append(f"  Risque Modere           : {n_modere:>5,} employes — Suivi mensuel")
lignes.append(f"  Risque Faible           : {n_faible:>5,} employes — Profil stable")
lignes.append("")
lignes.append("3. ANALYSE PAR DEPARTEMENT")
lignes.append("-" * 70)
for dept, risque in dept_risque.sort_values(ascending=False).items():
    n_dept = len(df[df["Department"]==dept])
    priorite = " <- PRIORITE" if dept == dept_max else ""
    lignes.append(f"  {dept:<35} : {risque:.1f}% (n={n_dept}){priorite}")
lignes.append("")
lignes.append(f"  Dept le plus a risque : {dept_max} ({dept_risque[dept_max]:.1f}%)")
lignes.append(f"  Dept le plus stable   : {dept_min} ({dept_risque[dept_min]:.1f}%)")
lignes.append("")
lignes.append("4. FACTEURS DE RISQUE")
lignes.append("-" * 70)
lignes.append(f"  Avec heures sup  : {ot_yes:.1f}% de departs")
lignes.append(f"  Sans heures sup  : {ot_no:.1f}% de departs")
lignes.append(f"  Difference       : +{ot_yes - ot_no:.1f} points")
lignes.append("")
lignes.append("5. RECOMMANDATIONS STRATEGIQUES")
lignes.append("-" * 70)
lignes.append(f"  1. Entretiens immediats pour les {n_crit} employes critiques")
lignes.append(f"  2. Reduire les heures sup dans le dept {dept_max}")
lignes.append("  3. Programme de promotion interne")
lignes.append("  4. Ameliorer l equilibre vie pro/perso")
lignes.append("")
lignes.append("6. PERFORMANCES DU MODELE")
lignes.append("-" * 70)
lignes.append(f"  Algorithme      : XGBoost (GridSearchCV)")
lignes.append(f"  F1-Score        : {f1_final:.4f}")
lignes.append(f"  AUC-ROC         : {auc_final:.4f}")
lignes.append(f"  Seuil optimal   : {seuil:.2f}")
lignes.append("")
lignes.append("  Rapport genere par Generative HR Analytics")
lignes.append("  Seye Kine & Bindia Adeline Thiara | M. Aidara | UCAO 2025-2026")
lignes.append("=" * 70)

rapport = "\n".join(lignes)
print(rapport)

  RAPPORT RH — ANALYSE PREDICTIVE DU TURNOVER
  Genere automatiquement le 17/05/2026 a 11:38

1. RESUME EXECUTIF
----------------------------------------------------------------------
Analyse realisee sur 1,470 employes.
Taux d attrition observe : 16.1% (237 departs).
Modele XGBoost : F1=0.4821 | AUC=0.8031

2. REPARTITION DES NIVEAUX DE RISQUE
----------------------------------------------------------------------
  Risque Critique (>=70%) :   200 employes — Action immediate
  Risque Eleve   (50-70%) :     6 employes — Entretien sous 2 semaines
  Risque Modere           :    50 employes — Suivi mensuel
  Risque Faible           : 1,214 employes — Profil stable

3. ANALYSE PAR DEPARTEMENT
----------------------------------------------------------------------
  Sales                               : 20.1% (n=446) <- PRIORITE
  Human Resources                     : 17.5% (n=63)
  Research & Development              : 12.3% (n=961)

  Dept le plus a risque : Sales (20.1%)
  Dept le plus sta

---
## Cellule 5 — Sauvegarde du rapport

In [5]:
with open("rapport_rh_genai.txt", "w", encoding="utf-8") as f:
    f.write(rapport)
print("OK — rapport_rh_genai.txt sauvegarde")

OK — rapport_rh_genai.txt sauvegarde
